# Practical Work 3 — Data Preprocessing Pipeline

## Dataset: Titanic

### Objective

Build a preprocessing pipeline for numerical and categorical features using ColumnTransformer and Pipeline.

The main goal is to handle missing values, encode categorical features, scale numerical features and prevent data leakage.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
from pathlib import Path

data_path = Path("../data/titanic.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (891, 11)


,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  891 non-null    int64  
 1   pclass    891 non-null    int64  
 2   name      891 non-null    str    
 3   sex       891 non-null    str    
 4   age       714 non-null    float64
 5   sibsp     891 non-null    int64  
 6   parch     891 non-null    int64  
 7   ticket    891 non-null    str    
 8   fare      891 non-null    float64
 9   cabin     204 non-null    str    
 10  embarked  889 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 76.7 KB


## 1. Missing Values Analysis

Before preprocessing, we examine missing values in the dataset.

In [4]:
missing_values = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Percentage": (df.isnull().mean() * 100).round(2)
})

missing_values

,Missing Count,Percentage
survived,0,0.00
pclass,0,0.00
name,0,0.00
sex,0,0.00
age,177,19.87
sibsp,0,0.00
parch,0,0.00
ticket,0,0.00
fare,0,0.00
cabin,687,77.10


## 2. Feature Classification

We select features for the preprocessing pipeline.

Numerical features:
- age
- sibsp
- parch
- fare

Categorical features:
- pclass
- sex
- embarked

Target variable:
- survived

The columns name, ticket and cabin are excluded from this experiment. Name and ticket contain high-cardinality identifiers, while cabin has a very large proportion of missing values.

In [5]:
numerical_features = [
    "age",
    "sibsp",
    "parch",
    "fare"
]

categorical_features = [
    "pclass",
    "sex",
    "embarked"
]

X = df[numerical_features + categorical_features].copy()
y = df["survived"].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (891, 7)
Target shape: (891,)


## 3. Train-Test Split

We split the dataset before calculating any preprocessing statistics.

The training set is used to fit the preprocessing pipeline.
The test set is used only to evaluate the transformations.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (712, 7)
Testing features: (179, 7)
Training target: (712,)
Testing target: (179,)


## 4. Preprocessing Strategy

### Numerical Features

The median is selected to fill missing numerical values because it is less sensitive to extreme values than the mean.

StandardScaler is used to scale numerical features.

### Categorical Features

Missing categorical values are replaced with a separate category called "Unknown".

OneHotEncoder converts categorical values into numerical columns.

The parameter handle_unknown="ignore" allows the pipeline to process categories that were not present in the training set.

In [7]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical pipeline created.")

Numerical pipeline created.


In [8]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

print("Categorical pipeline created.")

Categorical pipeline created.


## 5. ColumnTransformer

ColumnTransformer applies different preprocessing strategies to numerical and categorical features.

Both transformations are combined into one preprocessing object.

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("ColumnTransformer created successfully.")

ColumnTransformer created successfully.


In [10]:
preprocessing_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor)
])

print(preprocessing_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'sibsp', 'parch',
                                                   'fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='Unknown',
                                                                                 strategy='constant')),
                                                                  ('onehot',
         

## 6. Fit on Training Data

The preprocessing pipeline is fitted only on the training dataset.

The test dataset is not used to calculate medians, scaling parameters or categories.

In [11]:
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

print("Training data before preprocessing:", X_train.shape)
print("Training data after preprocessing:", X_train_processed.shape)

Training data before preprocessing: (712, 7)
Training data after preprocessing: (712, 13)


In [12]:
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Testing data before preprocessing:", X_test.shape)
print("Testing data after preprocessing:", X_test_processed.shape)

Testing data before preprocessing: (179, 7)
Testing data after preprocessing: (179, 13)


## 7. Data Leakage Verification

We verify that the preprocessing pipeline was fitted only on training data.

The test data is transformed using the parameters learned from the training data.

In [13]:
print("Train shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)

assert X_train_processed.shape[1] == X_test_processed.shape[1]

print("Feature dimensions match!")

Train shape: (712, 13)
Test shape: (179, 13)
Feature dimensions match!


In [14]:
trained_medians = (
    preprocessing_pipeline
    .named_steps["preprocessor"]
    .named_transformers_["num"]
    .named_steps["imputer"]
    .statistics_
)

train_medians = X_train[numerical_features].median().values

print("Medians learned by pipeline:")
print(trained_medians)

print("\nMedians calculated from training data:")
print(train_medians)

print("\nAre they equal?", np.allclose(trained_medians, train_medians))

Medians learned by pipeline:
[28.5     0.      0.     14.4542]

Medians calculated from training data:
[28.5     0.      0.     14.4542]

Are they equal? True


In [15]:
feature_names = (
    preprocessing_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print("Number of processed features:", len(feature_names))
print("\nFirst 15 feature names:")
print(feature_names[:15])

Number of processed features: 13

First 15 feature names:
['num__age' 'num__sibsp' 'num__parch' 'num__fare' 'cat__pclass_1'
 'cat__pclass_2' 'cat__pclass_3' 'cat__sex_female' 'cat__sex_male'
 'cat__embarked_C' 'cat__embarked_Q' 'cat__embarked_S'
 'cat__embarked_Unknown']


## 8. Conclusion

In this practical work, a preprocessing pipeline was created using Pipeline and ColumnTransformer.

The following steps were completed:

1. Numerical and categorical features were identified.
2. Missing numerical values were filled using the median.
3. Missing categorical values were replaced with "Unknown".
4. Numerical features were scaled using StandardScaler.
5. Categorical features were encoded using OneHotEncoder.
6. The dataset was split into training and testing sets before preprocessing.
7. The pipeline was fitted only on training data.
8. The test data was transformed without fitting the pipeline again.
9. The transformed training and testing datasets were checked for matching feature dimensions.
10. Data leakage was checked by comparing the learned medians with training data statistics.

The preprocessing pipeline successfully transforms numerical and categorical features while preventing data leakage.